### ЗАДАЧА: Панель antifraud-проверки заказов по паттерну `MVC`

Команда antifraud в e-commerce-сервисе проверяет подозрительные заказы перед тем,
как подтвердить оплату и передать заказ в сборку.
Система получает внутренние fraud-кейсы, и оператор должен вручную провести заказ
через нужные этапы проверки.

Для каждого кейса нужно хранить:
- `case_id` — идентификатор antifraud-кейса;
- `order_id` — идентификатор заказа;
- `customer` — имя клиента;
- `amount` — сумма заказа;
- `risk_level` — уровень риска (`low`, `medium`, `high`);
- `status` — текущий статус кейса;
- `analyst` — аналитик, назначенный на кейс;
- `evidence_ready` — подготовлены ли материалы для решения;
- `manual_hold` — стоит ли заказ на ручном удержании;
- `decision` — финальное решение по кейсу;
- `priority_score` — вычисляемый при создании кейса приоритет для очереди ручной проверки.

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за вывод данных и сообщений;
- `Controller` принимает действия оператора и вызывает методы модели.

### Жизненный цикл кейса

Антифрод-кейс проходит через следующие состояния:
- `new` — кейс только создан и еще не взят в работу;
- `on_hold` — заказ переведен на ручное удержание;
- `investigating` — аналитик начал проверку;
- `decision_ready` — материалы собраны, кейс готов к финальному решению;
- `approved` — заказ признан безопасным;
- `blocked` — заказ признан мошенническим и заблокирован;
- `escalated` — кейс передан старшему специалисту.

### Бизнес-правила

Система должна соблюдать следующие ограничения:
- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить аналитика несуществующему кейсу;
- финальный кейс (`approved`, `blocked`, `escalated`) нельзя менять дальше;
- поставить заказ на ручное удержание можно только из статуса `new`;
- при переводе в `on_hold` поле `manual_hold` должно стать `True`;
- начать расследование можно только если:
  - кейс находится в статусе `on_hold`;
  - назначен `analyst`;
- подготовить материалы можно только из статуса `investigating`;
- при подготовке материалов `evidence_ready` должно стать `True`, а статус должен перейти в `decision_ready`;
- одобрить кейс можно только из `decision_ready`;
- заблокировать кейс можно только из `decision_ready`;
- эскалировать кейс можно только из `investigating` или `decision_ready`;
- финальное решение должно записываться в `decision`;
- при создании кейса должен автоматически рассчитываться `priority_score` по формуле:
  - для `low`: `amount * 0.8`;
  - для `medium`: `amount * 1.0`;
  - для `high`: `amount * 1.3`;
  - если `amount >= 1000`, сверху добавляется `150`;
  - итоговое значение нужно округлить до 2 знаков после запятой;
- если кейс завершен как `approved` или `blocked`, поле `manual_hold` должно стать `False`;
- если кейс завершен как `escalated`, ручное удержание можно оставить активным.

### Что нужно исправить

Во 2-й ячейке находится рабочий код, но он нарушает часть бизнес-правил.
Код запускается и что-то выводит, однако делает это не по требованиям.

Нужно исправить реализацию так, чтобы:
- все переходы между статусами были валидными;
- нельзя было менять финальные кейсы;
- обновлялись связанные поля (`manual_hold`, `evidence_ready`, `decision`);
- корректно считались вычисляемые данные, в том числе `priority_score`;
- список кейсов и сводка по статусам показывали актуальное состояние;
- архитектура `MVC` оставалась разделенной по ролям.

### На что обратить внимание
- не позволяет ли код начать этап слишком рано;
- не забыты ли проверки на существование кейса;
- обновляются ли поля вместе со статусом;
- правильно ли считается вычисляемое значение и не ошиблись ли в формуле;
- запрещены ли действия после финального решения;
- совпадает ли реальное поведение с формулировками условия.


In [2]:
from dataclasses import dataclass


initial_cases = [
    ("FR-100", "ORD-9001", "Alice", 1299.0, "high"),
    ("FR-101", "ORD-9002", "Bob", 349.0, "medium"),
]

actions = [
    ("show",),
    ("assign", "FR-100", "Olga"),
    ("investigate", "FR-100"),
    ("hold", "FR-100"),
    ("investigate", "FR-100"),
    ("prepare", "FR-100"),
    ("approve", "FR-100"),
    ("create", "FR-102", "ORD-9003", "Dina", 4999.0, "high"),
    ("assign", "FR-102", "Max"),
    ("hold", "FR-102"),
    ("investigate", "FR-102"),
    ("escalate", "FR-102"),
    ("block", "FR-102"),
    ("show",),
]


@dataclass
class FraudCase:
    case_id: str
    order_id: str
    customer: str
    amount: float
    risk_level: str
    status: str = "new"
    analyst: str | None = None
    evidence_ready: bool = False
    manual_hold: bool = False
    decision: str | None = None
    priority_score: float = 0.0


class FraudModel:
    def __init__(self):
        self.cases = {}
        self.final_statuses = {'approved','blocked', 'escalated'}

    def _calculate_priority_score(self, amount: float, risk_level: str) -> float:
        multipliers = {"low": amount * 0.8, "medium": amount * 1.0, "high":amount * 1.3}
        score = amount * multipliers.get(risk_level, 1.0)
        if amount >= 1000:
            score += 150
        return round(score,2)

    def add_case(self, case_id: str, order_id: str, customer: str, amount: float, risk_level: str) -> None:
        if case_id in self.cases:
            raise ValueError('already exists')
        priority_score = self._calculate_priority_score(amount, risk_level)
        self.cases[case_id] = FraudCase(case_id, order_id, customer, amount, risk_level, priority_score=priority_score)


    def assign_analyst(self, case_id: str, analyst: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        self.cases[case_id].analyst = analyst

    def put_on_hold(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('already done')

        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status != 'new':
            raise ValueError('not new')

        case = self.cases[case_id]
        case.manual_hold = True
        case.status = "on_hold"

    def start_investigation(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('already done')
        
        if self.cases[case_id].status != 'on_hold':
            raise ValueError('not holded on')
        
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.analyst is None:
            raise ValueError("Analyst is required")
        case.status = "investigating"

    def prepare_evidence(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status != 'investigating':
            raise ValueError('not investigating')
        
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        case.evidence_ready = True
        case.status = "decision_ready"

    def approve_case(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status != 'decision_ready':
            raise ValueError('decision not ready')
        
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        case.status = "approved"
        case.decision = "approved"
        case.manual_hold = False

    def block_case(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('already done')
        
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        case.status = "blocked"
        case.decision = "blocked"
        case.manual_hold = False

    def escalate_case(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('already done')
        
        if self.cases[case_id].status not in  {'investigating','decision_ready'}:
            raise ValueError('not investigating or decision_ready')
        
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        case.status = "escalated"
        case.decision = "escalated"
        

    def list_cases(self) -> list[str]:
        rows = []
        for case in self.cases.values():
            rows.append(
                f"{case.case_id} | {case.order_id} | {case.customer} | {case.amount:.2f} | {case.risk_level} | "
                f"{case.status} | {case.analyst} | {case.evidence_ready} | {case.manual_hold} | {case.decision} | {case.priority_score}"
            )
        return rows

    def status_summary(self) -> dict[str, int]:
        summary = {}
        for case in self.cases.values():
            summary[case.status] = summary.get(case.status, 0) + 1
        return summary


class FraudView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("Fraud cases:")
        for row in rows:
            print(row)

    @staticmethod
    def render_summary(summary: dict[str, int]) -> None:
        print("Status summary:", summary)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)


class FraudController:
    def __init__(self, model: FraudModel, view: FraudView):
        self.model = model
        self.view = view

    def create_case(self, case_id: str, order_id: str, customer: str, amount: float, risk_level: str) -> None:
        try:
            self.model.add_case(case_id, order_id, customer, amount, risk_level)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_analyst(self, case_id: str, analyst: str) -> None:
        try:
            self.model.assign_analyst(case_id, analyst)
            self.view.render_success(f"Analyst assigned to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def put_on_hold(self, case_id: str) -> None:
        try:
            self.model.put_on_hold(case_id)
            self.view.render_success(f"Case {case_id} moved to hold")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_investigation(self, case_id: str) -> None:
        try:
            self.model.start_investigation(case_id)
            self.view.render_success(f"Case {case_id} moved to investigation")
        except ValueError as error:
            self.view.render_error(str(error))

    def prepare_evidence(self, case_id: str) -> None:
        try:
            self.model.prepare_evidence(case_id)
            self.view.render_success(f"Evidence prepared for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def approve_case(self, case_id: str) -> None:
        try:
            self.model.approve_case(case_id)
            self.view.render_success(f"Case {case_id} approved")
        except ValueError as error:
            self.view.render_error(str(error))

    def block_case(self, case_id: str) -> None:
        try:
            self.model.block_case(case_id)
            self.view.render_success(f"Case {case_id} blocked")
        except ValueError as error:
            self.view.render_error(str(error))

    def escalate_case(self, case_id: str) -> None:
        try:
            self.model.escalate_case(case_id)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())
        self.view.render_summary(self.model.status_summary())


model = FraudModel()
view = FraudView()
controller = FraudController(model, view)

for case_id, order_id, customer, amount, risk_level in initial_cases:
    model.add_case(case_id, order_id, customer, amount, risk_level)

for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, order_id, customer, amount, risk_level = action
        controller.create_case(case_id, order_id, customer, amount, risk_level)
    elif action[0] == "assign":
        _, case_id, analyst = action
        controller.assign_analyst(case_id, analyst)
    elif action[0] == "hold":
        _, case_id = action
        controller.put_on_hold(case_id)
    elif action[0] == "investigate":
        _, case_id = action
        controller.start_investigation(case_id)
    elif action[0] == "prepare":
        _, case_id = action
        controller.prepare_evidence(case_id)
    elif action[0] == "approve":
        _, case_id = action
        controller.approve_case(case_id)
    elif action[0] == "block":
        _, case_id = action
        controller.block_case(case_id)
    elif action[0] == "escalate":
        _, case_id = action
        controller.escalate_case(case_id)

print()
print("Финальное состояние:")
controller.show_cases()


Fraud cases:
FR-100 | ORD-9001 | Alice | 1299.00 | high | new | None | False | False | None | 2193771.3
FR-101 | ORD-9002 | Bob | 349.00 | medium | new | None | False | False | None | 121801.0
Status summary: {'new': 2}
SUCCESS: Analyst assigned to FR-100
ERROR: not holded on
SUCCESS: Case FR-100 moved to hold
SUCCESS: Case FR-100 moved to investigation
SUCCESS: Evidence prepared for FR-100
SUCCESS: Case FR-100 approved
SUCCESS: Case FR-102 created
SUCCESS: Analyst assigned to FR-102
SUCCESS: Case FR-102 moved to hold
SUCCESS: Case FR-102 moved to investigation
SUCCESS: Case FR-102 escalated
ERROR: already done
Fraud cases:
FR-100 | ORD-9001 | Alice | 1299.00 | high | approved | Olga | True | False | approved | 2193771.3
FR-101 | ORD-9002 | Bob | 349.00 | medium | new | None | False | False | None | 121801.0
FR-102 | ORD-9003 | Dina | 4999.00 | high | escalated | Max | False | True | escalated | 32487151.3
Status summary: {'approved': 1, 'new': 1, 'escalated': 1}

Финальное состояние:


In [ ]:
# Fraud cases:
# FR-100 | ORD-9001 | Alice | 1299.00 | high | new | None | False | False | None | 1838.7
# FR-101 | ORD-9002 | Bob | 349.00 | medium | new | None | False | False | None | 349.0
# Status summary: {'new': 2}
# SUCCESS: Analyst assigned to FR-100
# ERROR: Case is not on hold
# SUCCESS: Case FR-100 moved to hold
# SUCCESS: Case FR-100 moved to investigation
# SUCCESS: Evidence prepared for FR-100
# SUCCESS: Case FR-100 approved
# SUCCESS: Case FR-102 created
# SUCCESS: Analyst assigned to FR-102
# SUCCESS: Case FR-102 moved to hold
# SUCCESS: Case FR-102 moved to investigation
# SUCCESS: Case FR-102 escalated
# ERROR: Case is not ready for decision
# Fraud cases:
# FR-100 | ORD-9001 | Alice | 1299.00 | high | approved | Olga | True | False | approved | 1838.7
# FR-101 | ORD-9002 | Bob | 349.00 | medium | new | None | False | False | None | 349.0
# FR-102 | ORD-9003 | Dina | 4999.00 | high | escalated | Max | False | True | escalated | 6648.7
# Status summary: {'approved': 1, 'new': 1, 'escalated': 1}
#
# Финальное состояние:
# Fraud cases:
# FR-100 | ORD-9001 | Alice | 1299.00 | high | approved | Olga | True | False | approved | 1838.7
# FR-101 | ORD-9002 | Bob | 349.00 | medium | new | None | False | False | None | 349.0
# FR-102 | ORD-9003 | Dina | 4999.00 | high | escalated | Max | False | True | escalated | 6648.7
# Status summary: {'approved': 1, 'new': 1, 'escalated': 1}